# Musanga V3.1 — NB00: audited annotation update (v1 → Nadine v2)

**Purpose:** Compare `Circles_Annotations_Nadine_v2.shp` against the established v1 annotations and prepare a **versioned V3_1 input manifest** for rerunning NB01 onwards.

**No model training, no mosaics recomputed, and no overwrite of V3.** The original 15 MB V3 NB00 notebook and `Final/Outputs/V3` stay intact. We *inherit only the already audited V3 input registry*, then change the exhaustive circle annotation source after validation.

The audit reports: polygon counts and classes; geometry validity and repairs; assignment to the same 34 Nadine circles; duplicates; per-circle union canopy cover and annotation area; added/removed/changed outlines; site-independent change metrics.

**Stop/go gates:** missing inputs, invalid CRS, unexpected class coding, dangerous geometry repairs, or unassigned polygons stop the manifest export rather than silently accepting altered labels.

**Important:** V3 NB01 is pinned to v1 filename and schema. **Do not run old V3 NB01 with this manifest.** A separate V3_1 NB01 revision is needed next.


## 01 — Load V3 baseline manifest, both annotation versions and circle boundaries

In [ ]:
%pip -q install geopandas pyogrio shapely rasterio pandas matplotlib
from pathlib import Path
from collections import defaultdict
import hashlib, json, warnings
import numpy as np, pandas as pd, geopandas as gpd
from shapely import make_valid
from shapely.ops import unary_union
from shapely.geometry import Polygon, MultiPolygon
from IPython.display import display
from google.colab import drive
if not Path('/content/drive/MyDrive').exists(): drive.mount('/content/drive')

ROOT=Path('/content/drive/MyDrive/Colab Notebooks/Musanga/Final')
if not ROOT.exists():
    ROOT=Path('/content/drive/MyDrive/Musanga/Final')
assert ROOT.exists(), 'Musanga/Final not found'
DATA=ROOT/'Data'
V3_MANIFEST=ROOT/'Outputs/V3/00_input_audit/00_input_manifest.json'
OUT=ROOT/'Outputs/V3_1/00_input_audit'
OUT.mkdir(parents=True,exist_ok=True)
assert V3_MANIFEST.exists(), (
    'Baseline V3 manifest absent. Run V3 NB00 Block 10 once (read-only).')
base=json.loads(V3_MANIFEST.read_text())
assert base.get('schema_version')=='v3_input_audit_5_nadine_circles',base.get('schema_version')
ex=base['exhaustive_annotations']
V1=Path(ex['positive_polygons'])
CIRCLES_PATH=Path(ex['complete_roi_boundaries'])
if not V1.exists() or not CIRCLES_PATH.exists():
    raise FileNotFoundError('V3 original annotation/circle path missing; check V3 manifest.')
# Locate v2 near original first; refuse multiple ambiguous candidates.
candidates=[p for p in V1.parent.rglob('Circles_Annotations_Nadine_v2.shp') if p.is_file()]
if not candidates:
    candidates=list(DATA.rglob('Circles_Annotations_Nadine_v2.shp'))
assert len(candidates)==1, (
    'Expected exactly one v2 shapefile. Found: '+str(candidates))
V2=candidates[0]
print('READ ONLY v1:',V1)
print('READ ONLY v2:',V2)
print('CIRCLE BOUNDARIES:',CIRCLES_PATH)
print('V3_1 outputs:',OUT)

v1=gpd.read_file(V1).reset_index(drop=True)
v2=gpd.read_file(V2).reset_index(drop=True)
circles=gpd.read_file(CIRCLES_PATH).reset_index(drop=True)
assert v1.crs is not None and v2.crs is not None and circles.crs is not None
assert len(circles)==34, 'The reference registry assumes 34 original Nadine circle rows.'
assert len(v1)>0 and len(v2)>0
for d in (v1,v2):d['source_row']=np.arange(len(d),dtype=int)
circles['circle_id']=np.arange(len(circles),dtype=int)
print('V1:',len(v1),'polygons, CRS',v1.crs,'fields',list(v1.columns))
print('V2:',len(v2),'polygons, CRS',v2.crs,'fields',list(v2.columns))
print('Circle rows:',len(circles),'CRS',circles.crs)


## 02 — Attribute and geometry QA

V3's exhaustive annotation contract treats **all polygons as Musanga**, regardless of the legacy `Class_ID` code. We do **not** silently reinterpret codes as negatives in V3_1. However, inspect the v2 attribute distributions before accepting this contract; if v2 introduces confirmed non-Musanga classes, adjust downstream semantics explicitly and do not approve this manifest yet.

We repair invalid polygons in-memory only when the geometry is still a valid polygonal representation with ≤1% area change. Neither source shapefile is edited.


In [ ]:
def class_table(g,version):
    out=[]
    for column in [c for c in g.columns if any(k in c.lower() for k in
                  ('class','species','espece','musanga','label'))]:
        counts=g[column].astype('string').fillna('<missing>').value_counts(dropna=False)
        for value,n in counts.items():
            out.append(dict(version=version,field=column,value=str(value),n=int(n)))
    return out
classes=pd.DataFrame(class_table(v1,'v1')+class_table(v2,'v2'),
                     columns=['version','field','value','n'])
display(classes)
classes.to_csv(OUT/'00_annotation_class_fields.csv',index=False)

REPAIRS=[]
def safe_polygon(g,version,row):
    if g is None or g.is_empty:raise ValueError(f'{version} row {row}: empty geometry')
    if g.geom_type in ('Polygon','MultiPolygon') and g.is_valid:return g
    old_area=g.area
    changed=make_valid(g)
    polygons=[]
    def parts(h):
        if h.geom_type=='Polygon':polygons.append(h)
        elif h.geom_type=='MultiPolygon':polygons.extend(list(h.geoms))
        elif hasattr(h,'geoms'):
            for piece in h.geoms:parts(piece)
    parts(changed)
    if not polygons:raise ValueError(f'{version} row {row}: geometry lost polygon')
    revised=unary_union(polygons)
    ratio=abs(revised.area-old_area)/max(old_area,revised.area,1e-12)
    if ratio>0.01 or not revised.is_valid:
        raise ValueError(f'{version} row {row}: unsafe repair {ratio:.3%}')
    REPAIRS.append(dict(version=version,source_row=row,
                        original_type=g.geom_type,relative_area_change=ratio))
    return revised
v1.geometry=[safe_polygon(g,'v1',i) for i,g in enumerate(v1.geometry)]
v2.geometry=[safe_polygon(g,'v2',i) for i,g in enumerate(v2.geometry)]
repairs=pd.DataFrame(REPAIRS,columns=['version','source_row','original_type','relative_area_change'])
display(repairs)
repairs.to_csv(OUT/'00_geometry_repairs.csv',index=False)
print('REPAIRED:',len(repairs),'polygons; originals not modified.')


## 03 — Assign polygons to the same 34 circles; compare geometry and cover

Assignment uses the **largest polygon/circle intersection**, with a strict majority-of-area rule. Crown boundaries may legitimately cross the circle border; the polygon is preserved whole, but circle cover is calculated using its portion inside the circle.

Metric CRS is taken from an existing *projected* circle CRS; if source circles are geographic, automatically use the local estimated UTM CRS.


In [ ]:
METRIC_CRS=circles.crs if circles.crs.is_projected else circles.estimate_utm_crs()
assert METRIC_CRS is not None
v1m=v1.to_crs(METRIC_CRS).copy()
v2m=v2.to_crs(METRIC_CRS).copy()
cm=circles.to_crs(METRIC_CRS).copy()
circle_geom=list(cm.geometry)
def assign(gdf,version):
    assignment=[];fractions=[];unassigned=[]
    spatial=cm.sindex
    for i,g in enumerate(gdf.geometry):
        hits=list(spatial.query(g,predicate='intersects'))
        scores=sorted(((g.intersection(circle_geom[int(j)]).area,int(j))
                       for j in hits),reverse=True)
        if not scores or scores[0][0]<=0:
            unassigned.append((i,'no intersection'));assignment.append(-1);fractions.append(0.)
            continue
        overlap,cid=scores[0];frac=overlap/max(g.area,1e-9)
        if frac<0.5:unassigned.append((i,round(frac,3)))
        assignment.append(cid);fractions.append(frac)
    gdf['circle_id']=assignment;gdf['fraction_inside']=fractions
    if unassigned:
        display(pd.DataFrame(unassigned,columns=['source_row','reason']))
        raise ValueError(f'{version}: {len(unassigned)} polygon(s) not confidently assigned')
    return gdf
v1m=assign(v1m,'v1');v2m=assign(v2m,'v2')
area_rows=[]
for version,gdf in [('v1',v1m),('v2',v2m)]:
    for cid,circle in enumerate(circle_geom):
        group=gdf[gdf.circle_id==cid]
        clipped=[g.intersection(circle) for g in group.geometry if g.intersects(circle)]
        union=unary_union(clipped) if clipped else None
        raw_area=sum(g.area for g in clipped)
        covered_area=union.area if union is not None else 0.
        area_rows.append(dict(version=version,circle_id=cid,
            circle_area_m2=circle.area,polygons=len(group),
            summed_annotation_area_m2=raw_area,
            musanga_union_area_m2=covered_area,
            musanga_cover_pct=100*covered_area/max(circle.area,1e-9),
            overlapping_annotation_area_m2=max(0,raw_area-covered_area),
            outside_circle_polygons=int((group.fraction_inside<0.99).sum())))
area=pd.DataFrame(area_rows)
wide=area.pivot(index='circle_id',columns='version')
comparison=pd.DataFrame({'circle_id':range(34)})
for col in ['polygons','musanga_cover_pct','musanga_union_area_m2',
            'overlapping_annotation_area_m2','outside_circle_polygons']:
    for version in ['v1','v2']:
        comparison[col+'_'+version]=wide[col][version].to_numpy()
comparison['delta_polygons']=comparison.polygons_v2-comparison.polygons_v1
comparison['delta_cover_pp']=comparison.musanga_cover_pct_v2-comparison.musanga_cover_pct_v1
comparison['abs_delta_cover_pp']=comparison.delta_cover_pp.abs()
comparison['delta_union_area_m2']=comparison.musanga_union_area_m2_v2-comparison.musanga_union_area_m2_v1
comparison.to_csv(OUT/'00_circle_v1_v2_comparison.csv',index=False)
area.to_csv(OUT/'00_circle_reference_metrics_long.csv',index=False)
display(comparison.round(3))
print('Changes in count:',len(v2)-len(v1))
print('Circle mean |cover change| pp:',round(comparison.abs_delta_cover_pp.mean(),3))
print('Largest cover changes (pp):')
display(comparison.nlargest(10,'abs_delta_cover_pp')[['circle_id','polygons_v1','polygons_v2',
            'musanga_cover_pct_v1','musanga_cover_pct_v2','delta_cover_pp']].round(3))


## 04 — Shape changes, overlaps, and class/coverage gates

This block compares *unions* within each circle; the union IoU quantifies agreement between versions, and added/removed area identifies the direction of changes. Checks for duplicated polygon geometries in v2 are separate.

**Do not assume row indices represent matched polygons across versions.** Use spatial overlay instead.


In [ ]:
shape_rows=[];duplicates=[]
for cid,circle in enumerate(circle_geom):
    a=[g.intersection(circle) for g in v1m.loc[v1m.circle_id==cid,'geometry']]
    b=[g.intersection(circle) for g in v2m.loc[v2m.circle_id==cid,'geometry']]
    ua=unary_union(a) if a else Polygon()
    ub=unary_union(b) if b else Polygon()
    intersect=ua.intersection(ub).area
    union=ua.union(ub).area
    shape_rows.append(dict(circle_id=cid,
        union_IoU_v1_v2=intersect/union if union else 1.,
        added_m2=max(0,ub.area-intersect),
        removed_m2=max(0,ua.area-intersect),
        symmetric_difference_m2=ua.symmetric_difference(ub).area))
    group=v2m[v2m.circle_id==cid]
    items=list(zip(group.source_row,group.geometry))
    for j,(row1,g1) in enumerate(items):
        for row2,g2 in items[j+1:]:
            if not g1.intersects(g2):continue
            overlap=g1.intersection(g2).area
            iou=overlap/max(g1.area+g2.area-overlap,1e-9)
            if iou>=0.9999:
                duplicates.append(dict(circle_id=cid,source_row_1=int(row1),
                                       source_row_2=int(row2),iou=iou))
shape=pd.DataFrame(shape_rows)
duplicates=pd.DataFrame(duplicates,columns=['circle_id','source_row_1','source_row_2','iou'])
shape.to_csv(OUT/'00_circle_union_shape_change.csv',index=False)
duplicates.to_csv(OUT/'00_v2_duplicate_polygons.csv',index=False)
display(shape.round(3))
print('EXACT v2 DUPLICATES:',len(duplicates))
display(duplicates)

# Preserve both source and geometry sanity checks in the manifest.
assert set(v1m.circle_id)==set(range(34)), 'v1 does not cover all 34 circle IDs'
assert set(v2m.circle_id)==set(range(34)), 'v2 missing one or more previously exhaustive circles'
assert not (comparison.musanga_cover_pct_v2>100.001).any()
assert len(duplicates)==0, 'V2 contains exact duplicate polygons: inspect before approval.'
print('GEOMETRY and CIRCLE-COVERAGE QA PASSED.')


## 05 — Approve and save immutable V3_1 manifest

This does **not** mutate the V3 manifest or source shapefiles. A compatible-looking copy of the original input registry is stored under `Outputs/V3_1`, with explicit `schema_version`, source fingerprints and `reference_version`.

**Critical:** inspect class distributions. If v2 now distinguishes negative from positive crown annotations rather than remaining an all-Musanga polygon layer, set `APPROVE_ALL_V2_POLYGONS_AS_MUSANGA=False` and **do not proceed** until the reference-label generator is revised.


In [ ]:
APPROVE_ALL_V2_POLYGONS_AS_MUSANGA=False   # Manually set True ONLY after checking v2 attributes
WRITE_V3_1_MANIFEST=False                   # Set True after reviewing Sections 02–04

def sha256_file(path,buffer=1024*1024):
    h=hashlib.sha256()
    with open(path,'rb') as handle:
        while True:
            chunk=handle.read(buffer)
            if not chunk:break
            h.update(chunk)
    return h.hexdigest()
def shapefile_fingerprint(shp):
    parts={}
    for suffix in ('.shp','.shx','.dbf','.prj','.cpg'):
        part=shp.with_suffix(suffix)
        if part.exists():parts[part.name]={'bytes':part.stat().st_size,
                                           'sha256':sha256_file(part)}
    assert all(shp.with_suffix(s).exists() for s in ('.shp','.shx','.dbf','.prj'))
    return parts

if WRITE_V3_1_MANIFEST:
    assert APPROVE_ALL_V2_POLYGONS_AS_MUSANGA, (
        'Explicit approval of all-v2-annotations-as-Musanga is required. '
        'If labels contain negatives, revise the reference contract first.')
    assert len(duplicates)==0
    assert v1m.circle_id.nunique()==34 and v2m.circle_id.nunique()==34
    registry=json.loads(json.dumps(base))  # Copy for preservation; original remains unchanged.
    registry['schema_version']='v3_1_input_audit_nadine_v2'
    registry['parent_schema_version']=base['schema_version']
    registry['reference_version']='Nadine_v2'
    registry['exhaustive_annotations']['positive_polygons']=str(V2)
    registry['exhaustive_annotations']['musanga_polygons']=int(len(v2))
    registry['exhaustive_annotations']['all_annotation_polygons_musanga']=True
    registry['exhaustive_annotations']['verified_exhaustive_circle_count']=34
    registry['exhaustive_annotations']['verified_exhaustive_circle_rows']=list(range(34))
    registry['annotation_versioning']={
        'baseline_manifest':str(V3_MANIFEST),
        'baseline_annotations':str(V1),
        'new_annotations':str(V2),
        'v1_sha256':shapefile_fingerprint(V1),
        'v2_sha256':shapefile_fingerprint(V2),
        'v1_feature_count':int(len(v1)),
        'v2_feature_count':int(len(v2)),
        'v2_geometry_repairs':int((repairs.version=='v2').sum()),
        'audit_circle_comparison':str(OUT/'00_circle_v1_v2_comparison.csv'),
        'audit_union_geometry':str(OUT/'00_circle_union_shape_change.csv'),
        'audit_class_fields':str(OUT/'00_annotation_class_fields.csv')
    }
    destination=OUT/'00_input_manifest.json'
    if destination.exists():
        old=json.loads(destination.read_text())
        assert old.get('annotation_versioning',{}).get('v2_sha256')==registry['annotation_versioning']['v2_sha256'], (
            'Source v2 changed after a prior audit! Review version history before overwriting.')
    destination.write_text(json.dumps(registry,indent=2))
    print('APPROVED V3_1 MANIFEST:',destination)
    print('V3 manifest unchanged:',V3_MANIFEST)
    print('IMPORTANT: Next update V3_1 NB01, not V3 NB01.')
else:
    print('Audit only. Review classes/geometries, then explicitly approve and enable manifest export.')
